# Multi-Agent LLM Collaborative Writing
## Qwen3-0.6B × 2 + LoRA on Kaggle T4×2

**Approach**: Train two small LLM agents with role specialization:
- Agent A: Concise summarizer (extract key points)
- Agent B: Detailed writer (expand into full summary)

**Baselines** (aligned with CoMLRL): Single, Parallel, Sequential, Discussion

In [ ]:
!pip install -q transformers datasets peft bitsandbytes accelerate torch wandb trl

In [ ]:
# Clone the project code from git
import os
REPO_URL = "https://github.com/JKpink/term-end-project.git"

if not os.path.exists("term-end-project"):
    !git clone {REPO_URL}
else:
    %cd term-end-project
    !git pull origin master
    %cd ..

# Change to project dir and add to path
%cd term-end-project/project
import sys
sys.path.insert(0, "src")

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU count: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")
    print(f"    VRAM: {torch.cuda.get_device_properties(i).total_memory / 1024**3:.1f} GB")

## 1. Train Collaborative Agents
TLDR summarization task: Agent A writes concise (~220 chars) + Agent B writes detailed (2-3x longer)

In [ ]:
from train import train_collaborative

final_a, final_b = train_collaborative(
    model_name="Qwen/Qwen3-0.6B",
    output_dir="./outputs/collab_tldr",
    task="tldr",
    dataset_size=320,
    num_epochs=3,
    lora_r=8,
    lora_alpha=16,
    learning_rate=1e-4,
    use_4bit=True,
)

## 2. Quick Test — See Collaboration in Action

In [ ]:
from transformers import AutoTokenizer
from train import load_agent_with_lora, build_tldr_formatters, collaborative_inference

model_name = "Qwen/Qwen3-0.6B"
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

agent_a, _ = load_agent_with_lora(model_name, use_4bit=True, device_map="cuda:0")
agent_b, _ = load_agent_with_lora(model_name, use_4bit=True, device_map="cuda:1" if torch.cuda.device_count() > 1 else "cuda:0")

# Load trained LoRA weights
from peft import PeftModel
agent_a = PeftModel.from_pretrained(agent_a, "./outputs/collab_tldr/agent_a_final")
agent_b = PeftModel.from_pretrained(agent_b, "./outputs/collab_tldr/agent_b_final")

formatters = build_tldr_formatters(tokenizer)

test_prompt = "I've been working remotely for 3 years and struggling with work-life balance. I feel isolated and my productivity dropped. Any advice?"

result = collaborative_inference(
    test_prompt, agent_a, agent_b, tokenizer,
    formatters[0], formatters[1], max_new_tokens=256,
)

print("=" * 60)
print("Agent A (Concise TLDR):")
print(result["agent_a"])
print("\n" + "=" * 60)
print("Agent B (Detailed Summary):")
print(result["agent_b"])

In [ ]:
# Save models for download
!mkdir -p /kaggle/working/outputs
!cp -r ./outputs/collab_tldr /kaggle/working/outputs/
print("Models saved to /kaggle/working/outputs/collab_tldr")